    Загружаем Базу с данными по торговым инструментам из файла;
    Формирование DF с данными торговых инструментам;
    Получаем массив торговых инструментов c MetaTrader 5 server;
    Устанавливаем в массиве торговых инструментов фиксированные спреды;
    Обновление массива символов на MetaTrader 5 server;

In [ ]:
import numpy as np
import pandas as pd
import sys
import os
import json

current_dir = os.getcwd()                                               # Определяем путь к текущему файлу (где выполняется код)
parent_dir = os.path.dirname(current_dir)                               # Переход на уровень выше (fc_to_mt5_migrations)
print(f"Рабочая директория проекта {parent_dir}")
config_path = os.path.join(parent_dir, "directory_config.txt")          # Определяем путь к файлу конфигурации

directories = {}                                                        # Читаем конфигурационный файл и создаём словарь с путями
if os.path.exists(config_path):
    with open(config_path, "r", encoding="utf-8") as file:
        for line in file:
            line = line.split("#")[0].strip()  # Убираем комментарии и пробелы
            if "=" in line:
                key, value = map(str.strip, line.split("=", 1))
                directories[key] = os.path.join(parent_dir, value.strip("'\""))     # Формируем абсолютный путь
else: print(f"❌ ERROR: Файл конфигурации '{config_path}' не найден.")

for key, path in directories.items(): print(f"📂 {key}: {path}")                    # Вывод всех загруженных директорий

directory_data_temp_files   = directories["directory_data_temp_files"]
directory_data_log_files    = directories["directory_data_log_files"]
libraries_path = os.path.join(directories["directory_libraries_path"])          # Формируем путь к libraries_py каталогу с библиотеками *.py

sys.path.append(libraries_path)                                 # sys.path — это список путей, где Python ищет модули при import module_name.
if libraries_path in sys.path: print(f"✅ Каталог {libraries_path} успешно добавлен в sys.path")
else: print(f"❌ Ошибка: {libraries_path} не найден в sys.path")

# Динамически импорт необходимых функций <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
file_imports = "dynamic_import_functions.py"                            # Библиотека для динамического импорта
file_imports_path = os.path.join(libraries_path, file_imports)

if os.path.exists(file_imports_path):
    import importlib
    importlib.invalidate_caches()                                       # Сбрасываем кэш перед импортом
    from dynamic_import_functions import import_functions, print_import_function_info
    print(f"\n ✅ Импорт [{file_imports}] успешен.")
else: print(f"\n ERROR: Файл '{file_imports}' не найден по пути {file_imports_path}, импорт не выполнен.\n")

modules_to_import = {                                   # Формируем словарь, с именами файлов и функциями в них
    "yar_sed_general_lib":
        [libraries_path,

                "pd_set_option",                        # Вывод ДФ
                "CSVLoader",
                "move_column",
                ],                           # Загрузка ДФ из CSV
                
    "mt5_api":
        [libraries_path,
                #"mt_5_manager",
                "mt5manager",
                #"manager_connect_with_control",
                #"manager_disconnect_with_control",
                "admin_connect_with_control",
                "admin_disconnect_with_control",
                "getting_array_trading_instruments"]              
                    }

imported = import_functions(modules_to_import)          # Импортируем модули из словаря modules_to_import
print_import_function_info(modules_to_import, imported) # Выводим переменные ожидаемые импортированными функциями 

In [ ]:
# [ОБЯЗАТЕЛЕН] указываем имя файла с данными торговых инструментов <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
# ``````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````
file_name = 'customers_symbol_crm_mt5_df.csv'     # CSV Файл с данными по символам

# [ОБЯЗАТЕЛЕН] Загружаем Базу с данными по торговым инструментамБ <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
# ``````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````
file_path = os.path.join(directory_data_temp_files, file_name)                              

csv_loader = imported["CSVLoader"](file_path, delimiter=',', encoding='ISO-8859-1', df_name='symbol_mapinr_df')     # вызываем класс  Создаём ДФ из CSV Файла
symbol_crm_df = csv_loader.load_data()  
symbol_crm_df["symbol_fix_spread"] = ((symbol_crm_df["buy_last_value"] - symbol_crm_df["sell_last_value"])*10**symbol_crm_df["symbol_digits_mt5"]).astype(int)  # Приводим к типу float
symbol_crm_df.loc[symbol_crm_df["symbol_fix_spread"] == 0, "symbol_fix_spread"] = 1


# меняем местами колонки в ДФ <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
list_col_name = ["mapping", "symbol_digits_mt5",  "symbol_digits", "symbol_fix_spread"]
symbol_crm_df = imported['move_column'](symbol_crm_df, list_col_name, new_position=0, new_position_step=1)

# Проверка: есть ли повторы
duplicates = symbol_crm_df[symbol_crm_df.duplicated(subset="mapping", keep=False)]


mismatch_rows = symbol_crm_df[symbol_crm_df["symbol_digits_mt5"] != symbol_crm_df["symbol_digits"]]
if mismatch_rows.empty:
    print("Все значения совпадают.")
else:
    print("\n symbol_crm_df[symbol_digits_mt5] symbol_crm_df[symbol_digits] Нужно проверит все строки и вывести те в которых значения не совпадают:")
    imported["pd_set_option"]("DF с несовпадением", mismatch_rows, 3) # Выводим строки с несовпадением


if not duplicates.empty:
    print("Найдены повторяющиеся значения в колонке 'mapping':")
    print(duplicates)
else: print("Повторов [ symbol_crm_df[mapping] ]нет — все значения уникальны.")

imported["pd_set_option"]("DF с данными торговых инструментов", symbol_crm_df, 3)

DF с несовпадением
<div>
<style scoped>
    .dataframe tbody tr th:only-of-type {
        vertical-align: middle;
    }

    .dataframe tbody tr th {
        vertical-align: top;
    }

    .dataframe thead th {
        text-align: right;
    }
</style>
<table border="1" class="dataframe">
  <thead>
    <tr style="text-align: right;">
      <th></th>
      <th>mapping</th>
      <th>symbol_digits_mt5</th>
      <th>symbol_digits</th>
      <th>symbol_fix_spread</th>
      <th>symbol_contract_size_mt5</th>
      <th>available_group_clients</th>
      <th>currency_name</th>
      <th>symbol_contract_size</th>
      <th>currency_id</th>
      <th>trading_group_id</th>
      <th>parent_currency_id</th>
      <th>currency_key_lr</th>
      <th>_symbol</th>
      <th>symbol_description</th>
      <th>symbol_profit</th>
      <th>symbol_margin</th>
      <th>symbol_point</th>
      <th>symbol_point_tech</th>
      <th>symbol_multiply</th>
      <th>symbol_calc_mode</th>
      <th>symbol_margin_initial</th>
      <th>0_Open</th>
      <th>0_Close</th>
      <th>1_Open</th>
      <th>1_Close</th>
      <th>2_Open</th>
      <th>2_Close</th>
      <th>3_Open</th>
      <th>3_Close</th>
      <th>4_Open</th>
      <th>4_Close</th>
      <th>5_Open</th>
      <th>5_Close</th>
      <th>6_Open</th>
      <th>6_Close</th>
      <th>leverage</th>
      <th>spread</th>
      <th>swap_buy</th>
      <th>swap_sell</th>
      <th>minimum_change</th>
      <th>commission</th>
      <th>expiration</th>
      <th>sell_last_value</th>
      <th>buy_last_value</th>
      <th>asset_id</th>
      <th>asset_priority</th>
      <th>asset_active</th>
      <th>updated_at</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <th>10</th>
      <td>XAUUSD</td>
      <td>3</td>
      <td>2</td>
      <td>329</td>
      <td>100.0</td>
      <td>True</td>
      <td>GOLD</td>
      <td>100.0</td>
      <td>74</td>
      <td>1</td>
      <td>74</td>
      <td>GOLD</td>
      <td>GOLD</td>
      <td>Gold vs US-Dollar</td>
      <td>USD</td>
      <td>USD</td>
      <td>0.01</td>
      <td>0.001</td>
      <td>1000.0</td>
      <td>2</td>
      <td>0.01</td>
      <td>0</td>
      <td>0</td>
      <td>0</td>
      <td>1439</td>
      <td>0</td>
      <td>1439</td>
      <td>0</td>
      <td>1439</td>
      <td>0</td>
      <td>1439</td>
      <td>0</td>
      <td>1439</td>
      <td>0</td>
      <td>0</td>
      <td>100</td>
      <td>4500</td>
      <td>0</td>
      <td>0</td>
      <td>NaN</td>
      <td>NaN</td>
      <td>NaN</td>
      <td>3284.59</td>
      <td>3284.92</td>
      <td>2</td>
      <td>1</td>
      <td>1</td>
      <td>2025-04-30 15:20:14</td>
    </tr>
  </tbody>
</table>
</div>
Повторов [ symbol_crm_df[mapping] ]нет — все значения уникальны.

DF с данными торговых инструментов
<div>
<style scoped>
    .dataframe tbody tr th:only-of-type {
        vertical-align: middle;
    }

    .dataframe tbody tr th {
        vertical-align: top;
    }

    .dataframe thead th {
        text-align: right;
    }
</style>
<table border="1" class="dataframe">
  <thead>
    <tr style="text-align: right;">
      <th></th>
      <th>mapping</th>
      <th>symbol_digits_mt5</th>
      <th>symbol_digits</th>
      <th>symbol_fix_spread</th>
      <th>symbol_contract_size_mt5</th>
      <th>available_group_clients</th>
      <th>currency_name</th>
      <th>symbol_contract_size</th>
      <th>currency_id</th>
      <th>trading_group_id</th>
      <th>parent_currency_id</th>
      <th>currency_key_lr</th>
      <th>_symbol</th>
      <th>symbol_description</th>
      <th>symbol_profit</th>
      <th>symbol_margin</th>
      <th>symbol_point</th>
      <th>symbol_point_tech</th>
      <th>symbol_multiply</th>
      <th>symbol_calc_mode</th>
      <th>symbol_margin_initial</th>
      <th>0_Open</th>
      <th>0_Close</th>
      <th>1_Open</th>
      <th>1_Close</th>
      <th>2_Open</th>
      <th>2_Close</th>
      <th>3_Open</th>
      <th>3_Close</th>
      <th>4_Open</th>
      <th>4_Close</th>
      <th>5_Open</th>
      <th>5_Close</th>
      <th>6_Open</th>
      <th>6_Close</th>
      <th>leverage</th>
      <th>spread</th>
      <th>swap_buy</th>
      <th>swap_sell</th>
      <th>minimum_change</th>
      <th>commission</th>
      <th>expiration</th>
      <th>sell_last_value</th>
      <th>buy_last_value</th>
      <th>asset_id</th>
      <th>asset_priority</th>
      <th>asset_active</th>
      <th>updated_at</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <th>0</th>
      <td>AUDUSD</td>
      <td>5</td>
      <td>5</td>
      <td>9</td>
      <td>100000.0</td>
      <td>True</td>
      <td>AUD/USD</td>
      <td>100000.0</td>
      <td>10</td>
      <td>1</td>
      <td>10</td>
      <td>AUDUSD</td>
      <td>AUDUSD</td>
      <td>Australian Dollar vs US Dollar</td>
      <td>USD</td>
      <td>AUD</td>
      <td>0.01</td>
      <td>0.001</td>
      <td>100000.0</td>
      <td>0</td>
      <td>0.0</td>
      <td>0</td>
      <td>0</td>
      <td>0</td>
      <td>1440</td>
      <td>0</td>
      <td>1440</td>
      <td>0</td>
      <td>1440</td>
      <td>0</td>
      <td>1440</td>
      <td>0</td>
      <td>1440</td>
      <td>0</td>
      <td>0</td>
      <td>400</td>
      <td>3</td>
      <td>0</td>
      <td>0</td>
      <td>NaN</td>
      <td>NaN</td>
      <td>NaN</td>
      <td>0.6393</td>
      <td>0.63939</td>
      <td>1</td>
      <td>0</td>
      <td>1</td>
      <td>2025-04-30 15:20:14</td>
    </tr>
    <tr>
      <th>...</th>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
      <td>...</td>
    </tr>
    <tr>
      <th>34</th>
      <td>BTBT.OQ</td>
      <td>2</td>
      <td>2</td>
      <td>1</td>
      <td>100.0</td>
      <td>True</td>
      <td>Bit Digital, Inc.</td>
      <td>100.0</td>
      <td>864</td>
      <td>1</td>
      <td>864</td>
      <td>BTBT</td>
      <td>BTBT</td>
      <td>NaN</td>
      <td>USD</td>
      <td>USD</td>
      <td>0.01</td>
      <td>0.010</td>
      <td>100.0</td>
      <td>2</td>
      <td>0.1</td>
      <td>0</td>
      <td>0</td>
      <td>810</td>
      <td>1200</td>
      <td>810</td>
      <td>1200</td>
      <td>810</td>
      <td>1200</td>
      <td>810</td>
      <td>1200</td>
      <td>810</td>
      <td>1200</td>
      <td>0</td>
      <td>0</td>
      <td>10</td>
      <td>4</td>
      <td>0</td>
      <td>0</td>
      <td>NaN</td>
      <td>NaN</td>
      <td>NaN</td>
      <td>2.0300</td>
      <td>2.03000</td>
      <td>3</td>
      <td>0</td>
      <td>1</td>
      <td>2025-04-30 14:15:37</td>
    </tr>
  </tbody>
</table>
<p>35 rows × 48 columns</p>
</div>

In [ ]:
# Получаем массив торговых инструментов
symbol_array = imported["getting_array_trading_instruments"]('*')

if symbol_array:
    print(dir(symbol_array[0]))  # Покажет доступные атрибуты объекта

In [ ]:
# Устанавливаем в массиве торговых инструментов фиксированных спредов  <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
# ``````````````````````````````````````````````````````````````````````````````````````````````````````````````````````````
for a in symbol_array:
    symbol_name = a.Symbol
    print(a.Symbol, a.Spread)
    # Проверка наличия символа в колонке 'mapping'
    match = symbol_crm_df[symbol_crm_df["mapping"] == symbol_name]
    
    if not match.empty:
        # Получение значения из колонки 'symbol_fix_spread'
        fix_spread = int(match.iloc[0]["symbol_fix_spread"])
        print(f"fix_spread = {fix_spread}")
        a.Spread = fix_spread
        print(f"✅ {symbol_name}: fix_spread = {fix_spread}")
    else:
        print(f"❌ {symbol_name}: не найден в mapping")


In [ ]:
# Обновление массива символов на сервере <<<<<<<<<<<<<<<<<<<<<<<<<<<< 
admin = imported["admin_connect_with_control"](admin if 'admin' in locals() else None)
if admin:
    print("SymbolUpdateBatch = ", admin.SymbolUpdateBatch(symbol_array))
    print("manager.Disconnect() = ", admin.Disconnect())
else: print(f"MT5Admin Failed to connect to server: {MT5Manager.LastError()}")        # не удалось подключиться к серверу           

if imported["admin_disconnect_with_control"](admin): del admin
else: print("❌ ERROR: разъединение mt5admin c сервером НЕ удалось.")